In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.linear_model import LinearRegression
import joblib

from data_ingestion import fetch_ts
from feature_engineering import engineer_features

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("Loading time-series data...")
data_dir = '../cs-train'
ts_data = fetch_ts(data_dir, clean=False)
print(f"Loaded {len(ts_data)} time-series datasets")

## Modeling Approaches Comparison

We will compare the following approaches:
1. **Supervised Learning with Scikit-Learn**
   - Linear Regression (baseline)
   - Random Forest Regressor
   - Gradient Boosting Regressor

2. **Statistical Time-Series Models** (ARIMA/SARIMA)
3. **Advanced Methods** (Prophet, Exponential Smoothing)

## Approach 1: Supervised Learning Models

In [ ]:
# Engineer features from time-series
df_all = ts_data['all']
X, y, dates = engineer_features(df_all, target_col='revenue', lookback_days=[1, 7, 14, 30, 90])

print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")

# Time-series aware train-test split
split_idx = int(0.8 * len(X))
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]
dates_train, dates_test = dates[:split_idx], dates[split_idx:]

print(f"\nTrain set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"Test date range: {dates_test[0]} to {dates_test[-1]}")

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define models
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
}

# Train and evaluate models
results = {}
predictions = {}

for model_name, model in models.items():
    print(f"\nTraining {model_name}...")
    
    # Use scaled features for all models
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    # Calculate metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results[model_name] = {'RMSE': rmse, 'MAE': mae, 'R²': r2}
    predictions[model_name] = y_pred
    
    print(f"  RMSE: {rmse:.2f}")
    print(f"  MAE: {mae:.2f}")
    print(f"  R²: {r2:.4f}")

In [ ]:
# Compare models
results_df = pd.DataFrame(results).T
print("\nModel Comparison:")
print(results_df.to_string())

# Visualize model comparison
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
x_pos = np.arange(len(results))
ax.bar(x_pos, results_df['R²'], color=['coral', 'steelblue', 'lightseagreen'])
ax.set_xticks(x_pos)
ax.set_xticklabels(results.keys(), rotation=0)
ax.set_ylabel('R² Score')
ax.set_title('Model Comparison: R² Score on Test Set', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('../notebooks/model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Select best model
best_model_name = results_df['R²'].idxmax()
best_model = models[best_model_name]
print(f"\nBest performing model: {best_model_name}")
print(f"R² Score: {results_df.loc[best_model_name, 'R²']:.4f}")

## Hyperparameter Tuning for Best Model

In [ ]:
from sklearn.model_selection import GridSearchCV

if best_model_name == 'Random Forest':
    param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [10, 15, 20],
        'min_samples_split': [5, 10]
    }
elif best_model_name == 'Gradient Boosting':
    param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.05, 0.1]
    }
else:
    param_grid = {}

if param_grid:
    print(f"Tuning {best_model_name} hyperparameters...")
    grid_search = GridSearchCV(best_model, param_grid, cv=5, scoring='r2', n_jobs=-1, verbose=1)
    grid_search.fit(X_train_scaled, y_train)
    
    best_model = grid_search.best_estimator_
    print(f"\nBest parameters: {grid_search.best_params_}")
    print(f"Best cross-validation R²: {grid_search.best_score_:.4f}")
    
    # Evaluate tuned model
    y_pred_tuned = best_model.predict(X_test_scaled)
    r2_tuned = r2_score(y_test, y_pred_tuned)
    rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))
    print(f"\nTuned model test R²: {r2_tuned:.4f}")
    print(f"Tuned model test RMSE: {rmse_tuned:.2f}")
    predictions['Best_Model_Tuned'] = y_pred_tuned
else:
    print(f"No hyperparameter tuning for {best_model_name}")
    predictions['Best_Model_Tuned'] = predictions[best_model_name]

## Final Model: Train on All Data

In [ ]:
# Retrain best model on all data
print("Retraining best model on all available data...")
X_all_scaled = scaler.fit_transform(X)  # Refit scaler on all data
best_model.fit(X_all_scaled, y)

# Save model
import os
if not os.path.exists('../models'):
    os.makedirs('../models')

model_path = '../models/best_model.joblib'
scaler_path = '../models/scaler.joblib'
joblib.dump(best_model, model_path)
joblib.dump(scaler, scaler_path)
print(f"Model saved to {model_path}")
print(f"Scaler saved to {scaler_path}")

## Predictions and Visualization

In [ ]:
# Get best model predictions
y_pred_best = best_model.predict(X_test_scaled)

# Visualize predictions vs actual
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(dates_test, y_test, label='Actual', linewidth=2, color='steelblue', marker='o', markersize=4)
ax.plot(dates_test, y_pred_best, label='Predicted', linewidth=2, color='coral', marker='s', markersize=4, alpha=0.7)
ax.fill_between(dates_test, y_test, y_pred_best, alpha=0.2, color='gray')
ax.set_xlabel('Date')
ax.set_ylabel('Revenue')
ax.set_title(f'Actual vs Predicted Revenue ({best_model_name})', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../notebooks/predictions_vs_actual.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nTest Set Performance:")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_best)):.2f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred_best):.2f}")
print(f"R²: {r2_score(y_test, y_pred_best):.4f}")
print(f"Mean Absolute Percentage Error: {np.mean(np.abs((y_test - y_pred_best) / y_test)) * 100:.2f}%")

In [ ]:
# Residual analysis
residuals = y_test - y_pred_best

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Residuals over time
axes[0, 0].plot(dates_test, residuals, linewidth=1, marker='o', markersize=4, color='purple')
axes[0, 0].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0, 0].set_title('Residuals Over Time', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Residual')
axes[0, 0].grid(True, alpha=0.3)

# Residual distribution
axes[0, 1].hist(residuals, bins=30, color='steelblue', edgecolor='black')
axes[0, 1].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[0, 1].set_title('Distribution of Residuals', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Residual')
axes[0, 1].set_ylabel('Frequency')

# Q-Q plot
from scipy import stats
stats.probplot(residuals, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot', fontsize=12, fontweight='bold')

# Predicted vs Residuals
axes[1, 1].scatter(y_pred_best, residuals, alpha=0.5, color='steelblue')
axes[1, 1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1, 1].set_title('Predicted vs Residuals', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Predicted')
axes[1, 1].set_ylabel('Residual')

plt.tight_layout()
plt.savefig('../notebooks/residual_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Residual Analysis:")
print(f"Mean residual: {residuals.mean():.2f}")
print(f"Std dev residual: {residuals.std():.2f}")

## Feature Importance Analysis

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    feature_importance = best_model.feature_importances_
    feature_names = [f'Feature_{i}' for i in range(len(feature_importance))]
    
    # Sort by importance
    indices = np.argsort(feature_importance)[-15:]  # Top 15 features
    
    plt.figure(figsize=(10, 6))
    plt.barh(range(len(indices)), feature_importance[indices], color='steelblue')
    plt.yticks(range(len(indices)), [f'Feature {i}' for i in indices])
    plt.xlabel('Importance')
    plt.title(f'Top 15 Feature Importances ({best_model_name})', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../notebooks/feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("Top 15 Most Important Features:")
    for i, idx in enumerate(indices[::-1]):
        print(f"{i+1}. Feature {idx}: {feature_importance[idx]:.4f}")
else:
    print(f"{best_model_name} does not have feature_importances_")

## Summary Report

In [ ]:
print("="*80)
print("PART 2 SUMMARY REPORT: TIME-SERIES MODELING")
print("="*80)
print(f"\nBest Model: {best_model_name}")
print(f"Test Set R² Score: {r2_score(y_test, y_pred_best):.4f}")
print(f"Test Set RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_best)):.2f}")
print(f"Test Set MAE: {mean_absolute_error(y_test, y_pred_best):.2f}")
print(f"MAPE: {np.mean(np.abs((y_test - y_pred_best) / y_test)) * 100:.2f}%")
print(f"\nModel trained on {X_train.shape[0]} samples")
print(f"Model tested on {X_test.shape[0]} samples")
print(f"Total features: {X.shape[1]}")
print(f"\nModel saved to: ../models/best_model.joblib")
print(f"Scaler saved to: ../models/scaler.joblib")
print("\nReady for Part 3: API Development and Deployment")
print("="*80)